# Denoising Diffusion Probabilistic Models (DDPM) — From Scratch

> **Educational purpose only.** This notebook implements a DDPM-style diffusion model (a time-conditioned U-Net plus a Gaussian diffusion process) from scratch in PyTorch, purely to understand the mechanics behind diffusion models. It is **not** meant to be run end-to-end: training a model like this to produce good samples takes many GPU-hours over hundreds of epochs, which is well beyond what a laptop/CPU can do.

## Setup: imports, paths and utility functions

The next few cells import the required libraries, define where the dataset lives, and declare small helper functions used later on for plotting images, saving image grids to disk, building the CIFAR-10 dataloader, and creating the folders used to store checkpoints/results.

In [17]:
# Core dependencies: PyTorch for modeling/training, torchvision/PIL for image
# handling, tqdm for progress bars, and TensorBoard for logging metrics.
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch import optim
import logging
from torch.utils.tensorboard import SummaryWriter

In [18]:
# Local path where the CIFAR-10 dataset will be downloaded to / read from.
DATA_DIR = os.path.join("..", "data")

In [19]:
def plot_images(images: torch.Tensor) -> None:
    """Display a batch of images side by side in a single row.

    Args:
        images: Batch of images with shape (N, C, H, W).
    """
    plt.figure(figsize=(32, 32))
    plt.imshow(torch.cat(
        [
            torch.cat([i for i in images.cpu()], dim=1),
        ], dim=2).permute(1, 2, 0).cpu()
    )
    plt.show()

def save_images(images: torch.Tensor, path: str, **kwargs) -> None:
    """Arrange a batch of images into a grid and save it as a single image file.

    Args:
        images: Batch of images with shape (N, C, H, W).
        path: Destination file path for the saved grid image.
        **kwargs: Extra keyword arguments forwarded to `torchvision.utils.make_grid`.
    """
    grid = torchvision.utils.make_grid(images, **kwargs)
    ndarr = grid.permute(1, 2, 0).to('cpu').numpy()
    im = Image.fromarray(ndarr)
    im.save(path)


def get_data() -> DataLoader:
    """Build the CIFAR-10 dataloader used to train the diffusion model.

    Images are resized, randomly cropped to `image_size`, converted to tensors
    and normalized to the [-1, 1] range.

    Returns:
        A DataLoader yielding batches of (image, label) pairs from CIFAR-10.
    """
    transforms = torchvision.transforms.Compose(
        [
            torchvision.transforms.Resize(80),
            torchvision.transforms.RandomResizedCrop(64, scale=(0.8, 1.0)),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ]
    )
    dataset = torchvision.datasets.CIFAR10(DATA_DIR, download=True, transform=transforms)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    return dataloader

def setup_logging(run_name: str) -> None:
    """Create the folder structure used to store model checkpoints and result images.

    Args:
        run_name: Name identifying the current training run, used as a subfolder.
    """
    os.mkdirs("models", exists_ok=True)
    os.mkdirs("results", exists_ok=True)
    os.mkdirs(os.path.join("models", run_name), exists_ok=True)
    os.mkdirs(os.path.join("results", run_name), exists_ok=True)

## Hyperparameters & device

Training configuration (run name, epoch count, batch size, image size, learning rate) and the compute device to use. On this machine there is no CUDA/MPS-class GPU capable of running the full training loop, so these values are kept as in the original reference implementation purely for illustration — see the note above about this notebook being educational only.

In [20]:
# hyperparams

run_name = "DDPM_Unconditional"
epochs = 500
batch_size = 4
image_size = 64
lr = 3e-4
# Check if MPS is available
device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Using device: {device}")

Using device: mps


## Exponential Moving Average (EMA)

`EMA` keeps a smoothed ("shadow") copy of the model's weights during training. Instead of using the raw, noisy weights at the end of training to generate samples, diffusion models typically sample from this EMA copy because it tends to produce more stable, higher-quality results.

In [21]:
class EMA:
    """Exponential Moving Average tracker for a model's parameters.

    Maintains a shadow model whose weights are a decayed running average of
    the weights of the model being trained, which is typically used at
    inference/sampling time for more stable outputs.
    """

    def __init__(self, beta: float):
        """Initialize the EMA tracker.

        Args:
            beta: Decay factor in [0, 1) controlling how much weight is given
                to past parameters versus the current ones (closer to 1 means
                slower/smoother updates).
        """
        super().__init__()
        self.beta = beta
        self.step = 0

    def update_model_average(self, ma_model: nn.Module, current_model: nn.Module) -> None:
        """Update every parameter of the EMA model towards the current model's parameters.

        Args:
            ma_model: The EMA (shadow) model whose parameters are updated in place.
            current_model: The model currently being trained.
        """
        for current_params, ma_params in zip(current_model.parameters(), ma_model.parameters()):
            old_weight, up_weight = ma_params.data, current_params.data
            ma_params.data = self.update_average(old_weight, up_weight)

    def update_average(self, old: torch.Tensor, new: torch.Tensor) -> torch.Tensor:
        """Compute the exponentially-weighted average of an old and a new tensor.

        Args:
            old: Previous (EMA) value of the parameter tensor.
            new: Current value of the parameter tensor.

        Returns:
            The updated EMA value: `old` if it is None, otherwise the weighted average.
        """
        if old is None:
            return new
        return old * self.beta + (1 - self.beta) * new

    def step_ema(self, ema_model: nn.Module, model: nn.Module, step_start_ema: int = 2000) -> None:
        """Advance the EMA by one training step.

        Before `step_start_ema` steps have elapsed, the EMA weights are simply
        reset to match the trained model (warm-up), after which they follow
        the exponential moving average update rule.

        Args:
            ema_model: The EMA (shadow) model to update.
            model: The model currently being trained.
            step_start_ema: Number of steps to wait before starting to average.
        """
        if self.step < step_start_ema:
            self.reset_parameters(ema_model, model)
            self.step += 1
            return
        self.update_model_average(ema_model, model)
        self.step += 1

    def reset_parameters(self, ema_model: nn.Module, model: nn.Module) -> None:
        """Copy the trained model's weights directly into the EMA model.

        Args:
            ema_model: The EMA (shadow) model to reset.
            model: The model currently being trained.
        """
        ema_model.load_state_dict(model.state_dict())

## Self-Attention block

A standard multi-head self-attention block (with a pre-norm + feed-forward residual branch, à la a Transformer encoder layer) applied over the spatial positions of a feature map. It is inserted at the lower-resolution stages of the U-Net so the model can capture long-range spatial dependencies that convolutions alone would struggle with.

In [22]:
class SelfAttention(nn.Module):
    """Multi-head self-attention over the spatial positions of a feature map.

    Flattens a (C, size, size) feature map into a sequence of `size * size`
    tokens, applies multi-head self-attention with a residual connection,
    then a feed-forward residual branch, and reshapes back to the original
    spatial layout.
    """

    def __init__(self, channels: int, size: int):
        """Initialize the self-attention block.

        Args:
            channels: Number of channels in the input feature map.
            size: Spatial height/width of the (square) input feature map.
        """
        super(SelfAttention, self).__init__()
        self.channels = channels
        self.size = size
        self.mha = nn.MultiheadAttention(channels, 4, batch_first=True)
        self.ln = nn.LayerNorm([channels])
        self.ff_self = nn.Sequential(
            nn.LayerNorm([channels]),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply self-attention to a batch of feature maps.

        Args:
            x: Input feature map with shape (N, channels, size, size).

        Returns:
            Output feature map with the same shape as `x`.
        """
        x = x.view(-1, self.channels, self.size * self.size).swapaxes(1, 2)
        x_ln = self.ln(x)
        attention_value, _ = self.mha(x_ln, x_ln, x_ln)
        attention_value = attention_value + x
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.swapaxes(2, 1).view(-1, self.channels, self.size, self.size)

## U-Net building blocks

These classes are the reusable pieces the U-Net is assembled from:

- **`DoubleConv`** — a "Conv → GroupNorm → GELU" block applied twice, optionally with a residual connection.
- **`Down`** — a downsampling stage (max-pool + double conv) that also injects the timestep embedding into the feature map.
- **`Up`** — an upsampling stage (bilinear upsample + concatenation with the matching encoder skip connection + double conv) that likewise injects the timestep embedding.

In [23]:
class DoubleConv(nn.Module):
    """Two consecutive (Conv2d -> GroupNorm -> GELU) blocks, optionally residual.

    The second block omits the final GELU activation so that, when used
    residually, the activation is applied only once after the skip addition.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        mid_channels: int = None,
        residual: bool = False,
    ):
        """Initialize the double-convolution block.

        Args:
            in_channels: Number of input channels.
            out_channels: Number of output channels.
            mid_channels: Number of channels between the two convolutions.
                Defaults to `out_channels` when not provided.
            residual: If True, adds the input to the block's output before
                the final activation (requires in_channels == out_channels).
        """
        super().__init__()
        self.residual = residual
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(1, mid_channels),
            nn.GELU(),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(1, out_channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the double-convolution block.

        Args:
            x: Input feature map with shape (N, in_channels, H, W).

        Returns:
            Output feature map with shape (N, out_channels, H, W).
        """
        if self.residual:
            return F.gelu(x + self.double_conv(x))
        else:
            return self.double_conv(x)

In [24]:
class Down(nn.Module):
    """Downsampling stage of the U-Net: max-pool, double conv, then add timestep embedding."""

    def __init__(self, in_channels: int, out_channels: int, emb_dim: int = 256):
        """Initialize the downsampling block.

        Args:
            in_channels: Number of input channels.
            out_channels: Number of output channels.
            emb_dim: Dimensionality of the incoming timestep embedding.
        """
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, in_channels, residual=True),
            DoubleConv(in_channels, out_channels),
        )

        self.emb_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(
                emb_dim,
                out_channels
            ),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Downsample the input and inject the timestep embedding.

        Args:
            x: Input feature map with shape (N, in_channels, H, W).
            t: Timestep embedding with shape (N, emb_dim).

        Returns:
            Output feature map with shape (N, out_channels, H/2, W/2).
        """
        x = self.maxpool_conv(x)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        return x + emb

class Up(nn.Module):
    """Upsampling stage of the U-Net: bilinear upsample, skip-connect, double conv, then add timestep embedding."""

    def __init__(self, in_channels: int, out_channels: int, emb_dim: int = 256):
        """Initialize the upsampling block.

        Args:
            in_channels: Number of channels after concatenating with the skip connection.
            out_channels: Number of output channels.
            emb_dim: Dimensionality of the incoming timestep embedding.
        """
        super().__init__()

        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv = nn.Sequential(
            DoubleConv(in_channels, in_channels, residual=True),
            DoubleConv(in_channels, out_channels, in_channels // 2),
        )

        self.emb_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(
                emb_dim,
                out_channels
            ),
        )

    def forward(self, x: torch.Tensor, skip_x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Upsample the input, merge it with the encoder skip connection, and inject the timestep embedding.

        Args:
            x: Input feature map from the previous (lower-resolution) stage.
            skip_x: Matching feature map from the encoder path (skip connection).
            t: Timestep embedding with shape (N, emb_dim).

        Returns:
            Output feature map with shape (N, out_channels, H*2, W*2).
        """
        x = self.up(x)
        x = torch.cat([skip_x, x], dim=1)
        x = self.conv(x)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        return x + emb

## The U-Net backbones

`UNet` is the network trained to predict the noise present in a noisy image at a given timestep `t`. It uses a sinusoidal positional encoding (the same idea as in Transformers) to turn the scalar timestep into a vector embedding, then threads that embedding through every `Down`/`Up` stage.

`UNet_conditional` is the same architecture with an additional class-label embedding added to the timestep embedding, which lets the model be conditioned to generate a specific class of image (class-conditional generation) rather than an unconditional one.

In [25]:
class UNet(nn.Module):
    """Unconditional U-Net used to predict the noise added to an image at a given timestep."""

    def __init__(self, c_in: int = 3, c_out: int = 3, time_dim: int = 256, device: str = "cuda"):
        """Initialize the U-Net.

        Args:
            c_in: Number of input image channels.
            c_out: Number of output channels (predicted noise channels).
            time_dim: Dimensionality of the timestep embedding.
            device: Device the positional encoding tensors are created on.
        """
        super().__init__()
        self.device = device
        self.time_dim = time_dim
        self.inc = DoubleConv(c_in, 64)
        self.down1 = Down(64, 128)
        self.sa1 = SelfAttention(128, 32)
        self.down2 = Down(128, 256)
        self.sa2 = SelfAttention(256, 16)
        self.down3 = Down(256, 256)
        self.sa3 = SelfAttention(256, 8)

        self.bot1 = DoubleConv(256, 512)
        self.bot2 = DoubleConv(512, 512)
        self.bot3 = DoubleConv(512, 256)

        self.up1 = Up(512, 128)
        self.sa4 = SelfAttention(128, 16)
        self.up2 = Up(256, 64)
        self.sa5 = SelfAttention(64, 32)
        self.up3 = Up(128, 64)
        self.sa6 = SelfAttention(64, 64)
        self.outc = nn.Conv2d(64, c_out, kernel_size=1)

    def pos_encoding(self, t: torch.Tensor, channels: int) -> torch.Tensor:
        """Compute a sinusoidal positional encoding for a batch of timesteps.

        Args:
            t: Timestep values with shape (N, 1).
            channels: Dimensionality of the resulting embedding.

        Returns:
            Positional encoding tensor with shape (N, channels).
        """
        inv_freq = 1.0 / (
            10000
            ** (torch.arange(0, channels, 2, device=self.device).float() / channels)
        )
        pos_enc_a = torch.sin(t.repeat(1, channels // 2) * inv_freq)
        pos_enc_b = torch.cos(t.repeat(1, channels // 2) * inv_freq)
        pos_enc = torch.cat([pos_enc_a, pos_enc_b], dim=-1)
        return pos_enc

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Predict the noise present in a batch of noisy images.

        Args:
            x: Noisy input images with shape (N, c_in, H, W).
            t: Timestep of each image in the batch, with shape (N,).

        Returns:
            Predicted noise with shape (N, c_out, H, W).
        """
        t = t.unsqueeze(-1).type(torch.float)
        t = self.pos_encoding(t, self.time_dim)

        x1 = self.inc(x)
        x2 = self.down1(x1, t)
        x2 = self.sa1(x2)
        x3 = self.down2(x2, t)
        x3 = self.sa2(x3)
        x4 = self.down3(x3, t)
        x4 = self.sa3(x4)

        x4 = self.bot1(x4)
        x4 = self.bot2(x4)
        x4 = self.bot3(x4)

        x = self.up1(x4, x3, t)
        x = self.sa4(x)
        x = self.up2(x, x2, t)
        x = self.sa5(x)
        x = self.up3(x, x1, t)
        x = self.sa6(x)
        output = self.outc(x)
        return output

In [26]:
class UNet_conditional(nn.Module):
    """Class-conditional U-Net: adds a class-label embedding on top of `UNet`.

    Identical to `UNet`, except that when a label `y` is provided its
    embedding is added to the timestep embedding before being propagated
    through the network, allowing the model to be conditioned on class.
    """

    def __init__(
        self,
        c_in: int = 3,
        c_out: int = 3,
        time_dim: int = 256,
        num_classes: int = None,
        device: str = "cuda",
    ):
        """Initialize the conditional U-Net.

        Args:
            c_in: Number of input image channels.
            c_out: Number of output channels (predicted noise channels).
            time_dim: Dimensionality of the timestep/label embedding.
            num_classes: Number of distinct class labels to condition on.
                If None, no label embedding layer is created.
            device: Device the positional encoding tensors are created on.
        """
        super().__init__()
        self.device = device
        self.time_dim = time_dim
        self.inc = DoubleConv(c_in, 64)
        self.down1 = Down(64, 128)
        self.sa1 = SelfAttention(128, 32)
        self.down2 = Down(128, 256)
        self.sa2 = SelfAttention(256, 16)
        self.down3 = Down(256, 256)
        self.sa3 = SelfAttention(256, 8)

        self.bot1 = DoubleConv(256, 512)
        self.bot2 = DoubleConv(512, 512)
        self.bot3 = DoubleConv(512, 256)

        self.up1 = Up(512, 128)
        self.sa4 = SelfAttention(128, 16)
        self.up2 = Up(256, 64)
        self.sa5 = SelfAttention(64, 32)
        self.up3 = Up(128, 64)
        self.sa6 = SelfAttention(64, 64)
        self.outc = nn.Conv2d(64, c_out, kernel_size=1)

        if num_classes is not None:
            self.label_emb = nn.Embedding(num_classes, time_dim)

    def pos_encoding(self, t: torch.Tensor, channels: int) -> torch.Tensor:
        """Compute a sinusoidal positional encoding for a batch of timesteps.

        Args:
            t: Timestep values with shape (N, 1).
            channels: Dimensionality of the resulting embedding.

        Returns:
            Positional encoding tensor with shape (N, channels).
        """
        inv_freq = 1.0 / (
            10000
            ** (torch.arange(0, channels, 2, device=self.device).float() / channels)
        )
        pos_enc_a = torch.sin(t.repeat(1, channels // 2) * inv_freq)
        pos_enc_b = torch.cos(t.repeat(1, channels // 2) * inv_freq)
        pos_enc = torch.cat([pos_enc_a, pos_enc_b], dim=-1)
        return pos_enc

    def forward(self, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """Predict the noise present in a batch of noisy images, conditioned on class labels.

        Args:
            x: Noisy input images with shape (N, c_in, H, W).
            t: Timestep of each image in the batch, with shape (N,).
            y: Class label of each image in the batch, with shape (N,).
                May be None to fall back to unconditional behavior.

        Returns:
            Predicted noise with shape (N, c_out, H, W).
        """
        t = t.unsqueeze(-1).type(torch.float)
        t = self.pos_encoding(t, self.time_dim)

        if y is not None:
            t += self.label_emb(y)

        x1 = self.inc(x)
        x2 = self.down1(x1, t)
        x2 = self.sa1(x2)
        x3 = self.down2(x2, t)
        x3 = self.sa2(x3)
        x4 = self.down3(x3, t)
        x4 = self.sa3(x4)

        x4 = self.bot1(x4)
        x4 = self.bot2(x4)
        x4 = self.bot3(x4)

        x = self.up1(x4, x3, t)
        x = self.sa4(x)
        x = self.up2(x, x2, t)
        x = self.sa5(x)
        x = self.up3(x, x1, t)
        x = self.sa6(x)
        output = self.outc(x)
        return output

## Sanity check

A quick smoke test: instantiate `UNet` on the CPU, count its parameters, and run a single forward pass on random dummy data to confirm the output shape matches the input image shape (as it must, since the network predicts noise of the same shape as the image). This is cheap enough to run on CPU and is only meant to verify the architecture is wired correctly — it performs no training.

In [27]:
if __name__ == '__main__':
    net = UNet(device="cpu")
    # net = UNet_conditional(num_classes=10, device="cpu")
    print(sum([p.numel() for p in net.parameters()]))
    x = torch.randn(3, 3, 64, 64)
    t = x.new_tensor([500] * x.shape[0]).long()
    y = x.new_tensor([1] * x.shape[0]).long()
    print(net(x, t).shape)

23332739
torch.Size([3, 3, 64, 64])


## The Diffusion process

The `Diffusion` class implements the actual DDPM math, independent of the network architecture:

- **Noise schedule** — a linear schedule of `beta` values controlling how much noise is added at each of the `noise_steps` timesteps, from which `alpha` and the cumulative product `alpha_hat` are derived.
- **Forward process (`noise_images`)** — given a clean image `x` and a timestep `t`, directly samples the noisy version `x_t` in closed form (no need to iterate step by step), plus the exact noise `epsilon` that was added — this is the training target.
- **Reverse process (`sample`)** — starting from pure Gaussian noise, iteratively applies the trained model's noise predictions to denoise the image step by step, back down to `t = 0`, producing a generated image.

In [28]:
class Diffusion:
    """Implements the DDPM forward (noising) and reverse (denoising/sampling) processes.

    Args:
        noise_steps: Number of steps in the diffusion process.
        beta_start: Starting value of beta (noise variance) in the schedule.
        beta_end: Ending value of beta (noise variance) in the schedule.
        img_size: Height/width of the (square) images being modeled.
        device: Device to run the diffusion computations on ('cpu' or 'cuda').
    """

    def __init__(
        self,
        noise_steps: int = 1000,
        beta_start: float = 1e-4,
        beta_end: float = 0.02,
        img_size: int = 256,
        device: str = "cuda",
    ):
        """Initialize the diffusion process and precompute its noise schedule."""
        self.noise_steps = noise_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.img_size = img_size
        self.device = device

        # Precompute the noise schedule and the diffusion-process parameters derived from it.
        self.beta = self.prepare_noise_schedule().to(device)
        self.alpha = 1. - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)

    def prepare_noise_schedule(self) -> torch.Tensor:
        """Build the linear noise (beta) schedule.

        Returns:
            A 1D tensor of length `noise_steps` linearly spaced between
            `beta_start` and `beta_end`.
        """
        return torch.linspace(self.beta_start, self.beta_end, self.noise_steps)

    def noise_images(self, x: torch.Tensor, t: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Sample noisy versions of a batch of images at the given timesteps.

        Uses the closed-form expression for the forward diffusion process, so
        the noisy image at timestep `t` can be sampled directly without
        iterating through all intermediate steps.

        Args:
            x: Clean input images with shape (N, C, H, W).
            t: Timestep for each image in the batch, with shape (N,).

        Returns:
            A tuple `(x_t, epsilon)`: the noisy images and the Gaussian noise
            that was added to produce them (the training target).
        """
        sqrt_alpha_hat = torch.sqrt(self.alpha_hat[t])[:, None, None, None]
        sqrt_one_minus_alpha_hat = torch.sqrt(1 - self.alpha_hat[t])[:, None, None, None]
        epsilon = torch.randn_like(x)
        return sqrt_alpha_hat * x + sqrt_one_minus_alpha_hat * epsilon, epsilon

    def sample_timesteps(self, n: int) -> torch.Tensor:
        """Sample random diffusion timesteps for a batch.

        Args:
            n: Number of timesteps to sample (typically the batch size).

        Returns:
            A tensor of shape (n,) with integer timesteps in [1, noise_steps).
        """
        return torch.randint(low=1, high=self.noise_steps, size=(n,))

    def sample(self, model: nn.Module, n: int) -> torch.Tensor:
        """Generate new images by running the reverse diffusion process.

        Args:
            model: Trained noise-prediction model (e.g. a `UNet`).
            n: Number of images to generate.

        Returns:
            A batch of `n` generated images as uint8 tensors with shape
            (n, 3, img_size, img_size) and values in [0, 255].
        """
        print(f"Sampling {n} new images....")
        # Switch to eval mode so the model does not train during sampling.
        model.eval()
        with torch.no_grad():
            # Start from pure Gaussian noise.
            x = torch.randn((n, 3, self.img_size, self.img_size)).to(self.device)
            # Iterate backwards over every noise step.
            for i in tqdm(reversed(range(1, self.noise_steps)), position=0):
                # Current timestep, broadcast to the whole batch.
                t = (torch.ones(n) * i).long().to(self.device)
                # The model predicts the noise present at this timestep.
                predicted_noise = model(x, t)
                alpha = self.alpha[t][:, None, None, None]
                alpha_hat = self.alpha_hat[t][:, None, None, None]
                beta = self.beta[t][:, None, None, None]
                if i > 1:
                    noise = torch.randn_like(x)
                else:
                    noise = torch.zeros_like(x)
                # Reverse process update: remove the predicted noise and add back a controlled amount of fresh noise.
                x = 1 / torch.sqrt(alpha) * (x - ((1 - alpha) / (torch.sqrt(1 - alpha_hat))) * predicted_noise) + torch.sqrt(beta) * noise
        # Restore training mode once sampling is done.
        model.train()
        x = (x.clamp(-1, 1) + 1) / 2
        x = (x * 255).type(torch.uint8)
        return x

## Training loop

`train()` ties everything together: for every batch it samples random timesteps, produces the corresponding noisy images via `Diffusion.noise_images`, asks the `UNet` to predict the noise that was added, and backpropagates the MSE between the predicted and true noise. At the end of each epoch it generates a batch of sample images with `Diffusion.sample` and checkpoints the model.

> **Reminder:** this is shown for completeness/understanding only. Running `train()` as-is would require a CUDA/MPS-capable GPU and a very long time (500 epochs over CIFAR-10) — it is not intended to be executed on CPU-only hardware.

In [29]:
def train() -> None:
    """Train the unconditional UNet diffusion model on CIFAR-10.

    Runs the full DDPM training loop for `epochs` epochs: for each batch it
    samples noisy images at random timesteps, has the model predict the
    added noise, and optimizes the MSE between predicted and true noise.
    After each epoch it samples example images and checkpoints the model.
    """
    # Build the CIFAR-10 dataloader.
    dataloader = get_data()
    # Create the model and move it to the target device.
    model = UNet(device=device).to(device)
    # Optimizer.
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    # Loss function.
    mse = nn.MSELoss()
    # Diffusion process instance.
    diffusion = Diffusion(img_size=image_size, device=device)
    logger = SummaryWriter(os.path.join("runs", run_name))
    l = len(dataloader)

    # Main training loop.
    for epoch in range(epochs):
        print(f"Starting epoch {epoch}:")
        pbar = tqdm(dataloader)
        for i, (images, _) in enumerate(pbar):
            # Move the batch of images to the target device.
            images = images.to(device)
            # Sample a random timestep t for each image in the batch.
            t = diffusion.sample_timesteps(images.shape[0]).to(device)
            # Produce the noisy image for the sampled timestep.
            x_t, noise = diffusion.noise_images(images, t)
            # Predict the added noise with the UNet.
            predicted_noise = model(x_t, t)
            # Compute the loss.
            loss = mse(noise, predicted_noise)

            # Optimization step.
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            pbar.set_postfix(MSE=loss.item())
            logger.add_scalar("MSE", loss.item(), global_step=epoch * l + i)

        # Sample example images at the end of each epoch (outside the training loop).
        sampled_images = diffusion.sample(model, n=images.shape[0])
        save_images(sampled_images, os.path.join("results", run_name, f"{epoch}.jpg"))
        torch.save(model.state_dict(), os.path.join("models", run_name, f"ckpt.pt"))